# 05 - Collaborative Filtering User-Based (KNN)

Recomenda produtos comprados por usuários com histórico similar (k-vizinhos mais próximos). Ver `docs/NOTEBOOKS.md` (seção 5) para a justificativa da amostragem de pool/avaliação.

## 0. Configuração Inicial

In [1]:
import json
import pickle
import random
import sys
from pathlib import Path

import mlflow
import numpy as np
import pandas as pd
import scipy.sparse as sp
import yaml
from sklearn.neighbors import NearestNeighbors

sys.path.insert(0, str(Path("..").resolve()))
from src.evaluation.metrics import evaluate_recommendations, pairs_to_ground_truth
from src.evaluation.ranking import recommendations_from_score_matrix

RANDOM_SEED = 42


def set_seed(seed: int) -> None:
    """Fixa a seed de aleatoriedade para reprodutibilidade."""
    random.seed(seed)
    np.random.seed(seed)


set_seed(RANDOM_SEED)

PROCESSED_DIR = Path("../data/processed")
MODELS_DIR = Path("../models/user_based_cf")
MODELS_DIR.mkdir(parents=True, exist_ok=True)

with open("../configs/model_config.yaml", encoding="utf-8") as f:
    config = yaml.safe_load(f)
K = config["evaluation"]["k"]
N_NEIGHBORS = config["user_based_cf"]["n_neighbors"]
NEIGHBOR_POOL_SIZE = config["user_based_cf"]["neighbor_pool_size"]
EVAL_SAMPLE_SIZE = config["user_based_cf"]["eval_sample_size"]

mlflow.set_tracking_uri(f"file:{(Path('..') / 'mlruns').resolve()}")
mlflow.set_experiment("user_based_cf")

print(f"K={K}, n_neighbors={N_NEIGHBORS}")
print(f"pool={NEIGHBOR_POOL_SIZE}, eval_sample={EVAL_SAMPLE_SIZE}")

K=10, n_neighbors=20
pool=20000, eval_sample=3000


## 1. Carregamento dos Artefatos de Preprocessing

In [2]:
with open(PROCESSED_DIR / "vocabularies.pkl", "rb") as f:
    vocab = pickle.load(f)

interactions_prior = sp.load_npz(PROCESSED_DIR / "interactions_prior.npz")
val_pairs = pd.read_pickle(PROCESSED_DIR / "val_pairs.pkl")
test_pairs = pd.read_pickle(PROCESSED_DIR / "test_pairs.pkl")

n_users, n_items = interactions_prior.shape
val_ground_truth = pairs_to_ground_truth(val_pairs)
test_ground_truth = pairs_to_ground_truth(test_pairs)

print(f"n_users={n_users:,} | n_items={n_items:,}")

n_users=126,408 | n_items=3,000


## 2. Escopo: Pool de Vizinhos e Amostra de Avaliação

O pool de vizinhos é amostrado **apenas** entre os usuários do split de treino (nunca de validação/teste), evitando que um usuário avaliado encontre a si mesmo como vizinho. A avaliação em si é feita sobre uma amostra fixa (seed 42) de validação/teste, para manter o custo de busca de vizinhos (`O(amostra x pool)`) viável.

In [3]:
all_idx = set(range(n_users))
train_idx = np.array(sorted(all_idx - set(val_ground_truth) - set(test_ground_truth)))

rng = np.random.default_rng(RANDOM_SEED)
pool_size = min(NEIGHBOR_POOL_SIZE, len(train_idx))
pool_indices = rng.choice(train_idx, size=pool_size, replace=False)

val_users = list(val_ground_truth.keys())
test_users = list(test_ground_truth.keys())
val_sample = rng.choice(
    val_users, size=min(EVAL_SAMPLE_SIZE, len(val_users)), replace=False
)
test_sample = rng.choice(
    test_users, size=min(EVAL_SAMPLE_SIZE, len(test_users)), replace=False
)

print(f"pool={len(pool_indices):,} | val_sample={len(val_sample):,}")
print(f"test_sample={len(test_sample):,}")

pool=20,000 | val_sample=3,000
test_sample=3,000


In [4]:
val_ground_truth_sample = {u: val_ground_truth[u] for u in val_sample}
test_ground_truth_sample = {u: test_ground_truth[u] for u in test_sample}

## 3. Ajuste do Modelo KNN

In [5]:
knn_model = NearestNeighbors(metric="cosine", algorithm="brute")
knn_model.fit(interactions_prior[pool_indices])

print("Modelo KNN ajustado sobre o pool de vizinhos.")

Modelo KNN ajustado sobre o pool de vizinhos.


## 4. Random Search de n_neighbors (Validação)

Busca aleatória sobre os candidatos de `n_neighbors` definidos em `configs/model_config.yaml`, avaliada na amostra de **validação**. O modelo KNN já ajustado é reaproveitado: `n_neighbors` é passado por chamada a `kneighbors(...)`, sem necessidade de reajustar o modelo a cada candidato.

In [6]:
def build_user_cf_recommendations(
    query_indices: np.ndarray,
    interactions: sp.csr_matrix,
    pool_indices: np.ndarray,
    knn_model: NearestNeighbors,
    n_neighbors: int,
    k: int,
) -> dict[int, list[int]]:
    """Gera recomendacoes por similaridade entre usuarios (KNN cosine).

    Args:
        query_indices: user_idx avaliados.
        interactions: matriz esparsa (n_users, n_items) de historico.
        pool_indices: user_idx do pool de vizinhos candidatos.
        knn_model: modelo NearestNeighbors ja ajustado sobre o pool.
        n_neighbors: numero de vizinhos considerados nesta chamada.
        k: tamanho do top-k recomendado.

    Returns:
        Mapeamento user_idx -> lista de item_idx recomendados.
    """
    distances, neighbor_positions = knn_model.kneighbors(
        interactions[query_indices], n_neighbors=n_neighbors
    )
    similarities = 1 - distances
    n_query, n_actual_neighbors = neighbor_positions.shape
    rows = np.repeat(np.arange(n_query), n_actual_neighbors)
    cols = neighbor_positions.flatten()
    weights = sp.csr_matrix(
        (similarities.flatten(), (rows, cols)), shape=(n_query, len(pool_indices))
    )
    scores = weights.dot(interactions[pool_indices]).toarray()
    return recommendations_from_score_matrix(list(query_indices), scores, k)


search_cfg = config["user_based_cf"]["search"]
candidates = search_cfg["n_neighbors_choices"]
n_trials = min(search_cfg["n_trials"], len(candidates))
search_rng = np.random.default_rng(RANDOM_SEED)
trial_neighbors = search_rng.choice(candidates, size=n_trials, replace=False)

search_results = []
for trial_idx, n_neighbors_trial in enumerate(trial_neighbors):
    n_neighbors_trial = int(n_neighbors_trial)
    trial_recs = build_user_cf_recommendations(
        val_sample, interactions_prior, pool_indices, knn_model, n_neighbors_trial, K
    )
    trial_metrics = evaluate_recommendations(trial_recs, val_ground_truth_sample, K)
    search_results.append({"n_neighbors": n_neighbors_trial, **trial_metrics})

    with mlflow.start_run(run_name=f"search_trial_{trial_idx}_k_{n_neighbors_trial}"):
        mlflow.log_param("n_neighbors", n_neighbors_trial)
        mlflow.log_metrics(
            {f"val_{name}": value for name, value in trial_metrics.items()}
        )

search_df = pd.DataFrame(search_results).sort_values("recall_at_k", ascending=False)
N_NEIGHBORS = int(search_df.iloc[0]["n_neighbors"])
print(search_df)
print(f"Melhor n_neighbors (validacao, amostra): {N_NEIGHBORS}")

   n_neighbors  precision_at_k  recall_at_k  ndcg_at_k  map_at_k
2           50        0.155433     0.245930   0.250499  0.149556
1           30        0.155367     0.244946   0.249438  0.149079
3           20        0.153867     0.244417   0.247036  0.147169
0           10        0.145300     0.229513   0.235991  0.140303
Melhor n_neighbors (validacao, amostra): 50


## 5. Geração de Recomendações Finais (Melhor n_neighbors)

In [7]:
val_recommendations = build_user_cf_recommendations(
    val_sample, interactions_prior, pool_indices, knn_model, N_NEIGHBORS, K
)
test_recommendations = build_user_cf_recommendations(
    test_sample, interactions_prior, pool_indices, knn_model, N_NEIGHBORS, K
)

## 6. Avaliação (na Amostra)

In [8]:
val_metrics = evaluate_recommendations(val_recommendations, val_ground_truth_sample, K)
test_metrics = evaluate_recommendations(
    test_recommendations, test_ground_truth_sample, K
)

print("Validação (amostra):", val_metrics)
print("Teste (amostra):", test_metrics)

Validação (amostra): {'precision_at_k': 0.15543333333333337, 'recall_at_k': 0.2459295903876134, 'ndcg_at_k': 0.2504989657190952, 'map_at_k': 0.14955582441001092}
Teste (amostra): {'precision_at_k': 0.15843333333333334, 'recall_at_k': 0.25571309427351724, 'ndcg_at_k': 0.2580873226707455, 'map_at_k': 0.15619960117997816}


## 7. Persistência e Rastreamento no MLflow

In [9]:
with open(PROCESSED_DIR / "split_meta.json", encoding="utf-8") as f:
    split_meta = json.load(f)

with open(MODELS_DIR / "knn_model.pkl", "wb") as f:
    pickle.dump({"model": knn_model, "pool_indices": pool_indices}, f)

metrics_payload = {
    "k": K,
    "n_neighbors": N_NEIGHBORS,
    "neighbor_pool_size": int(len(pool_indices)),
    "eval_sample_size": int(len(val_sample)),
    "search_results": search_results,
    "validation": val_metrics,
    "test": test_metrics,
}
with open(MODELS_DIR / "metrics.json", "w", encoding="utf-8") as f:
    json.dump(metrics_payload, f, indent=2)

with mlflow.start_run(run_name="user_based_cf_final"):
    mlflow.log_params(
        {
            "k": K,
            "n_neighbors": N_NEIGHBORS,
            "neighbor_pool_size": len(pool_indices),
            "eval_sample_size": len(val_sample),
            "dataset_hash": split_meta["dataset_hash"],
            "selected_via_random_search": True,
        }
    )
    mlflow.log_metrics({f"val_{name}": value for name, value in val_metrics.items()})
    mlflow.log_metrics({f"test_{name}": value for name, value in test_metrics.items()})
    mlflow.log_artifact(str(MODELS_DIR / "metrics.json"))

print("User-based CF avaliado e rastreado no MLflow.")

User-based CF avaliado e rastreado no MLflow.
